# Train CLF2 (Intent) — EVA

Labels include scheduling, learning, and behavioral intents.


In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" scikit-learn


In [ ]:
import json
from pathlib import Path
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


In [ ]:
# EDIT THIS
ROOT = Path('/content/drive/MyDrive/CLF_Improvement')
DATASET_PATH = ROOT / 'datasets/clf2/final/clf2.jsonl'
assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"
print('Dataset:', DATASET_PATH)


In [ ]:
def load_jsonl(path: Path):
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return rows

rows = load_jsonl(DATASET_PATH)
labels = sorted({r['label'] for r in rows})
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}
print('labels:', labels)

ds = Dataset.from_list([
    {'text': r['text'], 'label': label2id[r['label']]}
    for r in rows
]).train_test_split(test_size=0.1, seed=1337)
ds


In [ ]:
BASE_MODEL = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tok(batch['text'], truncation=True, max_length=64)

ds_tok = ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tok)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)


In [ ]:
def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro'),
    }

args = TrainingArguments(
    output_dir='/content/clf2_runs',
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['test'],
    tokenizer=tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
metrics = trainer.evaluate()
print(metrics)

# Confusion matrix
pred = trainer.predict(ds_tok['test'])
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)
cm = confusion_matrix(y_true, y_pred)
print('labels:', labels)
print(cm)


In [ ]:
# Save final model + label_map.json
FINAL_DIR = Path('/content/clf2_final')
FINAL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL_DIR))
tok.save_pretrained(str(FINAL_DIR))

with (FINAL_DIR / 'label_map.json').open('w', encoding='utf-8') as f:
    json.dump({'id2label': {str(k): v for k, v in id2label.items()}}, f, indent=2)

print('Saved:', FINAL_DIR)


In [ ]:
# Zip for download
import shutil
zip_path = shutil.make_archive('/content/final_model', 'zip', str(FINAL_DIR))
print('ZIP:', zip_path)
